In [1]:
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
import kagglehub

# Initialize Spark Session with Kafka packages
spark = (SparkSession.builder.appName("MyApp")
    .config(
        "spark.jars.packages", 
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0"
    )
    .master("local[*]")
    .getOrCreate()
)

/home/liemak363/repo/Big Data Lab/lab_1/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/16 09:09:31 WARN Utils: Your hostname, LAPTOP-2TOMM4HG, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/16 09:09:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/liemak363/repo/Big%20Data%20Lab/lab_1/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/liemak363/.ivy2.5.2/cache
The jars for the packages stored in: /home/liemak363/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kaf

In [5]:
# path is ~/.cache/kagglehub/datasets as default
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row
+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row


In [ ]:
# First, delete topics if they exist
admin_client = AdminClient({"bootstrap.servers": "localhost:9092"})
admin_client.delete_topics(["ratings", "movies", "tags"], operation_timeout=10)
topic_metadata = admin_client.list_topics(timeout=10)
print("Deleted if exists")

# Create new topics
new_topics = [
    NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="movies", num_partitions=1, replication_factor=3),
    NewTopic(topic="tags", num_partitions=1, replication_factor=3),
]

fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()  # The result itself is None
        print(f"Topic {topic} created")
    except Exception as e:
        print(f"Failed to create topic {topic}: {e}")

Deleted if exists
Topic ratings created
Topic movies created
Topic tags created


In [6]:
# Push the dataframes to kafka broker
df_ratings.selectExpr("to_json(struct(*)) AS value").write.format("kafka").option(
    "kafka.bootstrap.servers", "localhost:9092"
).option("topic", "ratings").save()

df_movies.selectExpr("to_json(struct(*)) AS value").write.format("kafka").option(
    "kafka.bootstrap.servers", "localhost:9092"
).option("topic", "movies").save()

df_tags.selectExpr("to_json(struct(*)) AS value").write.format("kafka").option(
    "kafka.bootstrap.servers", "localhost:9092"
).option("topic", "tags").save()

# Check if data is in Kafka topics.
# It is stored in binary format in the "value" column.
df_ratings_kafka = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "ratings")
    .load()
)

# Cast the binary value to a readable string
df_ratings_kafka = df_ratings_kafka.selectExpr("CAST(value AS STRING)")
df_ratings_kafka.show(1, truncate=False)

+-----------------------------------------------------------+
|value                                                      |
+-----------------------------------------------------------+
|{"userId":1,"movieId":1,"rating":4.0,"timestamp":964982703}|
+-----------------------------------------------------------+
only showing top 1 row


In [ ]:
# ----------------------------------------------------------------------------------------------

In [5]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# 1. Define the schemas based on the original CSV structure
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

# 2. Read raw binary data from Kafka and cast to String
raw_ratings = spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "ratings") \
    .option("startingOffsets", "earliest") \
    .load().selectExpr("CAST(value AS STRING)")

raw_tags = spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "tags") \
    .option("startingOffsets", "earliest") \
    .load().selectExpr("CAST(value AS STRING)")

# 3. Parse the JSON strings into beautiful, usable columns!
df_parsed_ratings = raw_ratings.withColumn("data", from_json(col("value"), ratings_schema)).select("data.*")
df_parsed_tags = raw_tags.withColumn("data", from_json(col("value"), tags_schema)).select("data.*")

df_parsed_ratings.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows


In [3]:
from pyspark.sql.functions import col, from_json, avg, count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# 1. Define schemas matching the JSON payloads from Kafka
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

# 2. Helper function to read and parse a Kafka topic
def get_parsed_kafka_topic(topic_name, schema):
    return spark.read.format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", topic_name) \
        .option("startingOffsets", "earliest") \
        .load() \
        .selectExpr("CAST(value AS STRING)") \
        .withColumn("data", from_json(col("value"), schema)) \
        .select("data.*")

# 3. Load the DataFrames
df_movies = get_parsed_kafka_topic("movies", movies_schema)
df_ratings = get_parsed_kafka_topic("ratings", ratings_schema)
df_tags = get_parsed_kafka_topic("tags", tags_schema)

In [7]:
# Group by movieId to get average rating and total number of ratings
top_movies_df = df_ratings.groupBy("movieId") \
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("rating_count")
    ) \
    .filter(col("rating_count") > 30) \
    .orderBy(col("avg_rating").desc()) \
    .limit(5)

# Join with movies table to get the titles for the report
q1_result = top_movies_df.join(df_movies, "movieId", "inner") \
    .select("movieId", "title", "avg_rating", "rating_count") \
    .orderBy(col("avg_rating").desc())

print("--- Q1: Top 5 Movies (>30 ratings) ---")
q1_result.show(truncate=False)

--- Q1: Top 5 Movies (>30 ratings) ---
+-------+--------------------------------+-----------------+------------+
|movieId|title                           |avg_rating       |rating_count|
+-------+--------------------------------+-----------------+------------+
|318    |Shawshank Redemption, The (1994)|4.429022082018927|317         |
|1204   |Lawrence of Arabia (1962)       |4.3              |45          |
|858    |Godfather, The (1972)           |4.2890625        |192         |
|2959   |Fight Club (1999)               |4.272935779816514|218         |
|1276   |Cool Hand Luke (1967)           |4.271929824561403|57          |
+-------+--------------------------------+-----------------+------------+



In [9]:
# Find the 5 worst tags based on average rating
worst5_tags = (
    df_tags.join(df_ratings, on=["userId", "movieId"])
    .groupBy("tag")
    .agg(avg("rating").alias("avg_rating"), count("rating").alias("rating_count"))
    .orderBy(col("avg_rating").asc())
    .limit(5)
)

worst5_tags.show(truncate=False)

+---------------+----------+------------+
|tag            |avg_rating|rating_count|
+---------------+----------+------------+
|Ghosts         |0.5       |1           |
|symbolic       |0.5       |1           |
|HORRIBLE ACTING|0.5       |1           |
|boring         |0.75      |2           |
|camp           |1.0       |1           |
+---------------+----------+------------+



In [8]:
# 1. Calculate the average rating for every movie
movie_avg_ratings = df_ratings.groupBy("movieId").agg(avg("rating").alias("movie_avg_rating"))

# 2. Join the movie averages with the tags
tags_with_ratings = df_tags.join(movie_avg_ratings, "movieId", "inner")

# 3. Group by tag, calculate the overall average, and grab the bottom 5
worst_tags_df = tags_with_ratings.groupBy("tag") \
    .agg(avg("movie_avg_rating").alias("tag_avg_rating")) \
    .orderBy(col("tag_avg_rating").asc()) \
    .limit(5)

print("--- Q2: 5 Worst Tags by Average Rating ---")
worst_tags_df.show(truncate=False)

# Save these 5 tags into a Python list so we can use them in Question 3
worst_5_tags_list = [row['tag'] for row in worst_tags_df.collect()]

--- Q2: 5 Worst Tags by Average Rating ---
+--------+------------------+
|tag     |tag_avg_rating    |
+--------+------------------+
|symbolic|0.5               |
|shark   |1.4166666666666667|
|stage   |1.75              |
|Tokyo   |2.0               |
|SNL     |2.1               |
+--------+------------------+



In [ ]:
# 1. Filter our previously joined DataFrame to ONLY include rows with the 5 worst tags
movies_with_worst_tags = tags_with_ratings.filter(col("tag").isin(worst_5_tags_list))

# 2. Get the unique movies, their tags, and their individual average ratings
q3_movie_breakdown = movies_with_worst_tags.join(df_movies, "movieId", "inner") \
    .select("tag", "title", "movie_avg_rating") \
    .distinct() \
    .orderBy("tag", "movie_avg_rating")

print("--- Q3 Part A: Individual Movies Associated With The Worst Tags ---")
q3_movie_breakdown.show(50, truncate=False)

# 3. Count how many UNIQUE movies are associated with each of the worst tags
q3_tag_movie_counts = movies_with_worst_tags.select("tag", "movieId").distinct() \
    .groupBy("tag") \
    .agg(count("movieId").alias("number_of_movies_with_this_tag"))

print("--- Q3 Part B: How Many Movies Have These Tags? ---")
q3_tag_movie_counts.show(truncate=False)

--- Q3 Part A: Individual Movies Associated With The Worst Tags ---


+--------+------------------------------+------------------+
|tag     |title                         |movie_avg_rating  |
+--------+------------------------------+------------------+
|SNL     |Night at the Roxbury, A (1998)|2.1               |
|Tokyo   |Walk, Don't Run (1966)        |2.0               |
|shark   |Jaws 3-D (1983)               |1.4166666666666667|
|stage   |Being Julia (2004)            |1.75              |
|symbolic|Begotten (1990)               |0.5               |
+--------+------------------------------+------------------+

--- Q3 Part B: How Many Movies Have These Tags? ---


+--------+------------------------------+
|tag     |number_of_movies_with_this_tag|
+--------+------------------------------+
|Tokyo   |1                             |
|shark   |1                             |
|SNL     |1                             |
|symbolic|1                             |
|stage   |1                             |
+--------+------------------------------+



%6|1775102264.564|FAIL|rdkafka#producer-1| [thrd:localhost:9192/2]: localhost:9192/2: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1775102264.759|FAIL|rdkafka#producer-1| [thrd:localhost:9192/2]: localhost:9192/2: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY, 1 identical error(s) suppressed)
